# 🚀 FORECASTING - INFERENCIA 2025

## 📚 1. CONFIGURACIÓN Y LIBRERÍAS

In [ ]:
import pandas as pd
import numpy as np
import holidays
from IPython.display import display

# Configuración de visualización
pd.set_option('display.max_columns', None)

## 📥 2. CARGA DE DATOS DE INFERENCIA

In [ ]:
path_inferencia = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(path_inferencia, parse_dates=['fecha'])
print(f"Dataset cargado: {inferencia_df.shape}")
display(inferencia_df.head())

## ⚙️ 3. PIPELINE DE TRANSFORMACIÓN

In [ ]:
# 3.1 Variables Temporales y Eventos
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5, 6])
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['semana_anio'] = inferencia_df['fecha'].dt.isocalendar().week.astype(int)

es_holidays = holidays.Spain(years=[2025])
inferencia_df['es_festivo'] = inferencia_df['fecha'].apply(lambda x: x in es_holidays)
inferencia_df['es_black_friday'] = (inferencia_df['fecha'] == '2025-11-28')
inferencia_df['es_cyber_monday'] = (inferencia_df['fecha'] == '2025-12-01')

# 3.2 Lags y Media Móvil
def create_lags(df):
    df = df.copy().sort_values(['producto_id', 'fecha'])
    for lag in range(1, 8):
        df[f'lag_{lag}'] = df.groupby(['producto_id', df['fecha'].dt.year])['unidades_vendidas'].shift(lag)
    df['rolling_mean_7'] = df.groupby(['producto_id', df['fecha'].dt.year])['unidades_vendidas'].transform(lambda x: x.rolling(window=7, min_periods=7).mean())
    return df

inferencia_df = create_lags(inferencia_df)

# 3.3 Precios y Competencia
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['descuento_porcentaje'] = (((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100).round(2)
inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1).round(2)
inferencia_df['ratio_precio'] = (inferencia_df['precio_venta'] / inferencia_df['precio_competencia']).round(4)
inferencia_df = inferencia_df.drop(columns=competidores)

print(f"Variables de ingeniería creadas. Columnas actuales: {len(inferencia_df.columns)}")

## 🏷️ 4. CODIFICACIÓN CATEGÓRICA Y ALINEACIÓN

In [ ]:
# 4.1 Preparar columnas para OHE
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'])

# 4.2 Lista maestra de columnas del modelo (75 columnas)
cols_model = ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'dia_semana', 'año', 'mes', 'dia_mes', 'es_fin_semana', 'es_festivo', 'es_black_friday', 'es_cyber_monday', 'trimestre', 'semana_anio', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'rolling_mean_7', 'descuento_porcentaje', 'precio_competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900', 'nombre_h_Domyos Kit Mancuernas 20kg', 'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad', 'nombre_h_Lotuscrafts Yoga Bolster', 'nombre_h_Manduka PRO Yoga Mat', 'nombre_h_Merrell Moab 2 GTX', 'nombre_h_New Balance Fresh Foam X 1080v12', 'nombre_h_Nike Air Zoom Pegasus 40', 'nombre_h_Nike Dri-FIT Miler', 'nombre_h_Puma Velocity Nitro 2', 'nombre_h_Quechua MH500', 'nombre_h_Reebok Floatride Energy 5', 'nombre_h_Reebok Professional Deck', 'nombre_h_Salomon Speedcross 5 GTX', 'nombre_h_Sveltus Kettlebell 12kg', 'nombre_h_The North Face Borealis', 'nombre_h_Trek Marlin 7', 'categoria_h_Fitness', 'categoria_h_Outdoor', 'categoria_h_Running', 'categoria_h_Wellness', 'subcategoria_h_Banco Gimnasio', 'subcategoria_h_Bandas Elásticas', 'subcategoria_h_Bicicleta Montaña', 'subcategoria_h_Bloque Yoga', 'subcategoria_h_Cojín Yoga', 'subcategoria_h_Esterilla Fitness', 'subcategoria_h_Esterilla Yoga', 'subcategoria_h_Mancuernas Ajustables', 'subcategoria_h_Mochila Trekking', 'subcategoria_h_Pesa Rusa', 'subcategoria_h_Pesas Casa', 'subcategoria_h_Rodillera Yoga', 'subcategoria_h_Ropa Montaña', 'subcategoria_h_Ropa Running', 'subcategoria_h_Zapatillas Running', 'subcategoria_h_Zapatillas Trail']

# Asegurar alineación
for col in cols_model:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

inferencia_df = inferencia_df[cols_model]
print(f"✅ Alineación completada. Columnas totales: {len(inferencia_df.columns)}")

## 💾 5. FILTRADO Y EXPORTACIÓN

In [ ]:
# Filtrar Noviembre
inferencia_df_final = inferencia_df[inferencia_df['fecha'].dt.month == 11].copy()

print(f"Forma final del dataset: {inferencia_df_final.shape}")

# Guardar
inferencia_df_final.to_csv('../data/processed/inferencia_df_transformado.csv', index=False)
display(inferencia_df_final.head())